# FL Compression Study — Colab Sweep

**Results save to Google Drive automatically — safe to close the tab and come back.**

## How to use (first time)
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run **Cell 1** (mount Drive)
3. Run **Cell 2** (install packages) — then **Restart session** when prompted
4. Re-run **Cell 1** (remount Drive after restart)
5. Run **Cell 3** (clone repo + setup symlinks)
6. Run experiment cells: **Cell 4** → **Cell 5** → **Cell 6** in order

## After disconnect / reopen
Re-run **Cell 1**, then **Cell 3** — then continue from where you left off.  
Completed runs are auto-skipped; partial runs resume from checkpoint.

## Results location on Drive
`My Drive / fl_compression_study / results /`  
View CSVs from any device at drive.google.com

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────────
# Run this every time you open the notebook.
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10_lr_decay', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints/flower_cifar10_adaptive', exist_ok=True)

print('Drive mounted.')
print(f'Results will persist at: {DRIVE_BASE}/results/')

In [ ]:
# ── Cell 2: Install packages ───────────────────────────────────────────────────
# Run once. After it finishes, RESTART RUNTIME, then re-run Cell 1 → Cell 3.
import subprocess, sys

subprocess.run(['apt-get', 'install', '-y', '-q', 'cmake', 'zlib1g-dev'], check=False)

# Install numpy first to pin the version before anything else loads
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2.0', '--force-reinstall'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'flwr[simulation]==1.9.0', 'pandas', 'seaborn',
                '--force-reinstall'], check=True)

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pysz'], check=False)
print(f"pysz: {'OK' if r.returncode == 0 else 'FAILED — SZ runs unavailable'}")

print()
print('=' * 55)
print('  DONE — now do:')
print('  1. Runtime → Restart session')
print('  2. Re-run Cell 1  (remount Drive)')
print('  3. Run Cell 3     (clone + setup symlinks)')
print('  4. Run experiment cells')
print('=' * 55)

In [ ]:
# ── Cell 3: Clone repo + setup Drive symlinks ─────────────────────────────────
# Run after every restart (takes ~10 seconds).
import os, sys

DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'
REPO_DIR   = '/content/fl-compression-study'
REPO       = 'https://github.com/ayananyways/fl-compression-study.git'

import subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO, REPO_DIR], check=True)
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
    print('Repo updated.')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

def _link(local, drive_path):
    os.makedirs(drive_path, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        import shutil
        for f in os.listdir(local):
            shutil.copy2(f'{local}/{f}', drive_path)
        shutil.rmtree(local)
    elif os.path.isfile(local):
        os.remove(local)
    os.symlink(drive_path, local)
    print(f'  linked: {local}')

_link(f'{REPO_DIR}/results',                              f'{DRIVE_BASE}/results')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10',           f'{DRIVE_BASE}/checkpoints/flower_cifar10')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10_lr_decay',  f'{DRIVE_BASE}/checkpoints/flower_cifar10_lr_decay')
_link(f'{REPO_DIR}/checkpoints/flower_cifar10_adaptive',  f'{DRIVE_BASE}/checkpoints/flower_cifar10_adaptive')

import flwr, numpy as np
print(f'flwr {flwr.__version__}  |  numpy {np.__version__}')
print('Ready — run experiment cells below.')

In [ ]:
# ── Cell 4: SZ main sweep
# sz_eb0.001 x3 seeds + sz_eb0.01 x3 seeds
import subprocess, sys, os

REPO_DIR = '/content/fl-compression-study'
os.chdir(REPO_DIR)

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False, cwd=REPO_DIR)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode}')

BASE = [
    sys.executable, f'{REPO_DIR}/fl-flower/run.py',
    '--dataset', 'cifar10', '--rounds', '200',
    '--num-clients', '10', '--alpha', '0.5', '--local-epochs', '2',
    '--output', f'{REPO_DIR}/results/resnet20_cifar10_main.csv',
]

print('=' * 55)
print('  SZ eb=0.001  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.001', '--seed', str(seed)])

print('=' * 55)
print('  SZ eb=0.01  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.01', '--seed', str(seed)])

print('\nSZ main sweep done.')
import pandas as pd
if os.path.exists(f'{REPO_DIR}/results/resnet20_cifar10_main.csv'):
    df = pd.read_csv(f'{REPO_DIR}/results/resnet20_cifar10_main.csv')
    print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 5: LR decay sweep
# fp32+cosine, quant8+cosine, sz+cosine  x3 seeds each  = 9 runs
import subprocess, sys, os

REPO_DIR = '/content/fl-compression-study'
os.chdir(REPO_DIR)

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False, cwd=REPO_DIR)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode}')

BASE = [
    sys.executable, f'{REPO_DIR}/fl-flower/run.py',
    '--dataset', 'cifar10', '--rounds', '200',
    '--num-clients', '10', '--alpha', '0.5', '--local-epochs', '2',
    '--output', f'{REPO_DIR}/results/resnet20_cifar10_lr_decay.csv',
    '--lr-decay',
]

print('=' * 55)
print('  fp32 + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'none', '--seed', str(seed)])

print('=' * 55)
print('  quant_8bit + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'quantization', '--bits', '8', '--seed', str(seed)])

print('=' * 55)
print('  SZ eb=0.001 + cosine LR  x3 seeds')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--compressor', 'sz', '--error-bound', '0.001', '--seed', str(seed)])

print('\nLR decay sweep done.')
import pandas as pd
if os.path.exists(f'{REPO_DIR}/results/resnet20_cifar10_lr_decay.csv'):
    df = pd.read_csv(f'{REPO_DIR}/results/resnet20_cifar10_lr_decay.csv')
    print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 6: Adaptive SZ sweep
# Schedule A, B, C, A+cosine  x3 seeds each  = 12 runs
# Schedule C is the novel contribution (plateau-triggered error bound)
import subprocess, sys, os

REPO_DIR = '/content/fl-compression-study'
os.chdir(REPO_DIR)

def run(cmd):
    print(f"\n>>> {' '.join(str(c) for c in cmd)}")
    r = subprocess.run(cmd, check=False, cwd=REPO_DIR)
    if r.returncode != 0:
        print(f'WARNING: exit {r.returncode}')

BASE = [
    sys.executable, f'{REPO_DIR}/fl-flower/run.py',
    '--dataset', 'cifar10', '--rounds', '200',
    '--num-clients', '10', '--alpha', '0.5', '--local-epochs', '2',
    '--output', f'{REPO_DIR}/results/resnet20_cifar10_adaptive.csv',
]

for schedule in ['A', 'B', 'C']:
    print('=' * 55)
    print(f'  Schedule {schedule}  x3 seeds')
    print('=' * 55)
    for seed in [0, 1, 2]:
        run(BASE + ['--schedule', schedule, '--seed', str(seed)])

print('=' * 55)
print('  Schedule A + cosine LR  x3 seeds  (control)')
print('=' * 55)
for seed in [0, 1, 2]:
    run(BASE + ['--schedule', 'A', '--lr-decay', '--seed', str(seed)])

print('\nAdaptive sweep done.')
import pandas as pd
if os.path.exists(f'{REPO_DIR}/results/resnet20_cifar10_adaptive.csv'):
    df = pd.read_csv(f'{REPO_DIR}/results/resnet20_cifar10_adaptive.csv')
    print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))

In [ ]:
# ── Cell 7: Final summary
import pandas as pd, os

REPO_DIR   = '/content/fl-compression-study'
DRIVE_BASE = '/content/drive/MyDrive/fl_compression_study'

files = [
    (f'{REPO_DIR}/results/resnet20_cifar10_main.csv',     'Main sweep'),
    (f'{REPO_DIR}/results/resnet20_cifar10_lr_decay.csv', 'LR decay'),
    (f'{REPO_DIR}/results/resnet20_cifar10_adaptive.csv', 'Adaptive'),
]

for fpath, label in files:
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        done = df.groupby(['compressor', 'seed'])['round'].max()
        n_complete = (done >= 200).sum()
        print(f'\n=== {label} — {n_complete}/{len(done)} runs complete ===')
        print(df.groupby(['compressor', 'seed'])['round'].agg(['count', 'max']))
    else:
        print(f'\n{label}: not started')

print(f'\nAll results saved to Google Drive:')
print(f'  {DRIVE_BASE}/results/')
print('\nTo merge with local Mac results, download these CSVs and run:')
print('  python3 scripts/merge_results.py')